# Feature Ablation Preparation

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 70.


In [ ]:
# ============================================================
# FEATURE ABLATION — TRAIN-ONLY PREPROCESSING
# ============================================================
#
# NO TRAINING IN THIS CELL
#
# Creates THREE controlled feature caches:
#
#   1) Morgan-only
#   2) RDKit-only
#   3) Morgan + RDKit
#
# Important:
# - Exact original Interaction splits are preserved.
# - Drug1 -> Drug2 direction is preserved.
# - Interaction IDs / labels are untouched.
# - RDKit descriptor selection is fitted on TRAIN drugs ONLY.
# - RDKit imputation is fitted on TRAIN drugs ONLY.
# - RDKit scaling is fitted on TRAIN drugs ONLY.
# - Calibration/Test are transform-only.
# - Morgan requires no fitted scaling.
# - Old feature cache is READ-ONLY and is NOT overwritten.
#
# ============================================================

import os
import json
import hashlib
import shutil
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import AllChem, Descriptors

warnings.filterwarnings("ignore")


# ============================================================
# 1) PATHS
# ============================================================

ROOT = Path(
    "/kaggle/input/datasets/"
    "amiramohammeddd/ddi-v2-menna-projectt"
)

TRAIN_PATH = ROOT / "clean_train_split.csv"
CAL_PATH   = ROOT / "clean_calibration_split.csv"
TEST_PATH  = ROOT / "clean_test_split.csv"

OLD_CACHE = ROOT / "exact_v2_feature_cache_2249.npz"
OLD_META  = ROOT / "exact_v2_feature_meta.json"

OUT = Path(
    "/kaggle/working/"
    "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
)

OUT.mkdir(
    parents=True,
    exist_ok=True,
)


for path in [
    TRAIN_PATH,
    CAL_PATH,
    TEST_PATH,
    OLD_CACHE,
    OLD_META,
]:
    assert path.is_file(), f"Missing: {path}"


# ============================================================
# 2) HELPERS
# ============================================================

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def save_json(obj, path):
    path.write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


# ============================================================
# 3) LOAD EXACT ORIGINAL SPLITS
# ============================================================

splits = {
    "train": pd.read_csv(
        TRAIN_PATH,
        keep_default_na=False,
    ),
    "calibration": pd.read_csv(
        CAL_PATH,
        keep_default_na=False,
    ),
    "test": pd.read_csv(
        TEST_PATH,
        keep_default_na=False,
    ),
}


expected_rows = {
    "train": 134249,
    "calibration": 28768,
    "test": 28768,
}


for name, df in splits.items():

    assert len(df) == expected_rows[name], (
        name,
        len(df),
    )

    required_cols = {
        "Drug1",
        "Drug2",
        "Label",
        "label_id",
    }

    assert required_cols.issubset(
        df.columns
    ), (
        name,
        df.columns.tolist(),
    )

    df["label_id"] = (
        df["label_id"]
        .astype(int)
    )

    assert (
        df["label_id"].min() >= 0
        and
        df["label_id"].max() < 84
    )


print("=" * 90)
print("ORIGINAL INTERACTION SPLITS")
print("=" * 90)

for name, df in splits.items():
    print(
        f"{name:12s}: "
        f"{len(df):,} rows | "
        f"{df.label_id.nunique()} classes"
    )


# ============================================================
# 4) VERIFY IMMUTABLE 84-CLASS MAPPING
# ============================================================

mapping_records = []

for name, df in splits.items():

    tmp = (
        df[
            ["label_id", "Label"]
        ]
        .drop_duplicates()
    )

    mapping_records.append(tmp)


mapping_all = pd.concat(
    mapping_records,
    ignore_index=True,
).drop_duplicates()


counts_per_id = (
    mapping_all
    .groupby("label_id")["Label"]
    .nunique()
)


assert counts_per_id.eq(1).all()

assert set(
    mapping_all["label_id"]
) == set(range(84))


interaction_mapping = (
    mapping_all
    .sort_values("label_id")
    .reset_index(drop=True)
)


assert len(interaction_mapping) == 84


interaction_mapping.to_csv(
    OUT / "interaction_mapping_84_exact.csv",
    index=False,
)


print("\n Exact 84-class mapping verified.")


# ============================================================
# 5) USE HISTORICAL CACHE ORDER ONLY
# ============================================================
#
# IMPORTANT:
# We do NOT reuse old feature values.
# We only reuse all_drugs order for reproducible indexing.
#
# ============================================================

with np.load(
    OLD_CACHE,
    allow_pickle=True,
) as old:

    old_all_drugs = (
        old["all_drugs"]
        .astype(str)
    )

    old_shape = old["x"].shape


assert old_shape == (1705, 2249)
assert len(old_all_drugs) == 1705
assert len(set(old_all_drugs)) == 1705


split_drugs = set()

for df in splits.values():

    split_drugs.update(
        df["Drug1"]
        .astype(str)
        .tolist()
    )

    split_drugs.update(
        df["Drug2"]
        .astype(str)
        .tolist()
    )


assert split_drugs == set(
    old_all_drugs.tolist()
), (
    len(split_drugs),
    len(old_all_drugs),
)


all_drugs = old_all_drugs.copy()

drug_to_idx = {
    drug: i
    for i, drug in enumerate(all_drugs)
}


# ============================================================
# 6) TRAIN / CAL / TEST DRUG AUDIT
# ============================================================

def drug_set(df):

    return set(
        df["Drug1"]
        .astype(str)
    ) | set(
        df["Drug2"]
        .astype(str)
    )


train_drugs = drug_set(
    splits["train"]
)

cal_drugs = drug_set(
    splits["calibration"]
)

test_drugs = drug_set(
    splits["test"]
)


cal_unseen = (
    cal_drugs
    -
    train_drugs
)

test_unseen = (
    test_drugs
    -
    train_drugs
)


train_indices = np.array(
    sorted(
        drug_to_idx[d]
        for d in train_drugs
    ),
    dtype=np.int64,
)


print("\n" + "=" * 90)
print("DRUG FIT SCOPE")
print("=" * 90)

print(
    "Train drugs:",
    len(train_drugs),
)

print(
    "All drugs:",
    len(all_drugs),
)

print(
    "Calibration drugs unseen in Train:",
    len(cal_unseen),
)

print(
    "Test drugs unseen in Train:",
    len(test_unseen),
)


# Expected from the audit
assert len(train_drugs) == 1675
assert len(all_drugs) == 1705
assert len(cal_unseen) == 14
assert len(test_unseen) == 20


# ============================================================
# 7) LOAD HISTORICAL FEATURE DEFINITION
# ============================================================
#
# We use historical metadata ONLY to preserve the canonical
# descriptor NAME SET and Morgan definition.
#
# We do NOT use historical scaler means/stds.
# We do NOT use historical kept-mask.
#
# ============================================================

historical_meta = json.loads(
    OLD_META.read_text(
        encoding="utf-8"
    )
)


canonical_descriptor_names = (
    historical_meta[
        "rdkit_all_descriptor_names"
    ]
)


assert len(
    canonical_descriptor_names
) == 208


assert int(
    historical_meta["morgan_bits"]
) == 2048

assert int(
    historical_meta["morgan_radius"]
) == 2


MORGAN_BITS = 2048
MORGAN_RADIUS = 2


print("\nHistorical RDKit version:")
print(
    historical_meta.get(
        "rdkit_version"
    )
)

print("Current RDKit version:")
print(rdBase.rdkitVersion)

print(
    "Canonical RDKit descriptors:",
    len(canonical_descriptor_names),
)


# ============================================================
# 8) MAP CURRENT RDKIT FUNCTIONS TO THE SAME 208 NAMES
# ============================================================

current_descriptor_map = {
    name: func
    for name, func
    in Descriptors._descList
}


missing_descriptor_functions = [
    name
    for name in canonical_descriptor_names
    if name not in current_descriptor_map
]


assert not missing_descriptor_functions, (
    "Current RDKit is missing historical descriptors:\n"
    +
    "\n".join(
        missing_descriptor_functions
    )
)


descriptor_functions = [
    current_descriptor_map[name]
    for name in canonical_descriptor_names
]


# ============================================================
# 9) COMPUTE RAW MOLECULAR FEATURES
# ============================================================

n_drugs = len(all_drugs)

morgan = np.zeros(
    (n_drugs, MORGAN_BITS),
    dtype=np.float32,
)

rdkit_raw = np.full(
    (
        n_drugs,
        len(canonical_descriptor_names),
    ),
    np.nan,
    dtype=np.float64,
)


failed_molecules = []
descriptor_failure_counts = np.zeros(
    len(canonical_descriptor_names),
    dtype=np.int64,
)


print("\n" + "=" * 90)
print("COMPUTING RAW FEATURES")
print("=" * 90)


for i, smiles in enumerate(all_drugs):

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:

        failed_molecules.append(
            smiles
        )

        continue


    # --------------------------------------------------------
    # Morgan
    # Exact intended definition:
    # radius=2, 2048 bits, no chirality, no feature invariants.
    # --------------------------------------------------------

    fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        MORGAN_RADIUS,
        nBits=MORGAN_BITS,
        useChirality=False,
        useFeatures=False,
    )

    arr = np.zeros(
        MORGAN_BITS,
        dtype=np.int8,
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr,
    )

    morgan[i] = arr.astype(
        np.float32
    )


    # --------------------------------------------------------
    # RDKit raw descriptors
    # --------------------------------------------------------

    for j, func in enumerate(
        descriptor_functions
    ):

        try:

            value = float(
                func(mol)
            )

            if np.isfinite(value):
                rdkit_raw[i, j] = value

            else:
                descriptor_failure_counts[j] += 1

        except Exception:

            descriptor_failure_counts[j] += 1


assert not failed_molecules, (
    "Unparseable molecules found:\n"
    +
    "\n".join(
        failed_molecules[:20]
    )
)


assert np.isfinite(
    morgan
).all()


print(
    "Raw Morgan shape:",
    morgan.shape,
)

print(
    "Raw RDKit shape:",
    rdkit_raw.shape,
)

print(
    "Total raw RDKit non-finite values:",
    int(
        (~np.isfinite(rdkit_raw)).sum()
    ),
)


# ============================================================
# 10) TRAIN-ONLY RDKIT FIT
# ============================================================
#
# Fit steps:
#
#   A. For every descriptor:
#      estimate median using TRAIN drugs only.
#
#   B. Replace non-finite TRAIN values using TRAIN median.
#
#   C. Descriptor is retained only if TRAIN variance > 1e-12.
#
#   D. StandardScaler fitted ONLY on retained TRAIN matrix.
#
#   E. Same TRAIN median + scaler applied to all drugs.
#
# ============================================================

rdkit_train_raw = (
    rdkit_raw[
        train_indices
    ]
)


n_desc = len(
    canonical_descriptor_names
)


train_finite_count = np.zeros(
    n_desc,
    dtype=np.int64,
)

train_median = np.full(
    n_desc,
    np.nan,
    dtype=np.float64,
)

train_variance = np.full(
    n_desc,
    np.nan,
    dtype=np.float64,
)

keep_mask = np.zeros(
    n_desc,
    dtype=bool,
)


for j in range(n_desc):

    column = (
        rdkit_train_raw[:, j]
    )

    finite = np.isfinite(
        column
    )

    train_finite_count[j] = (
        finite.sum()
    )

    if finite.sum() == 0:
        continue


    median = np.median(
        column[finite]
    )

    train_median[j] = median


    train_imputed_col = (
        column.copy()
    )

    train_imputed_col[
        ~finite
    ] = median


    variance = np.var(
        train_imputed_col,
        ddof=0,
    )

    train_variance[j] = variance


    if (
        np.isfinite(variance)
        and
        variance > 1e-12
    ):
        keep_mask[j] = True


kept_names = [
    name
    for name, keep
    in zip(
        canonical_descriptor_names,
        keep_mask,
    )
    if keep
]


dropped_names = [
    name
    for name, keep
    in zip(
        canonical_descriptor_names,
        keep_mask,
    )
    if not keep
]


assert len(kept_names) > 0


# ============================================================
# 11) TRAIN-ONLY IMPUTATION
# ============================================================

rdkit_imputed = (
    rdkit_raw.copy()
)


for j in range(n_desc):

    if not keep_mask[j]:
        continue

    bad = ~np.isfinite(
        rdkit_imputed[:, j]
    )

    rdkit_imputed[
        bad,
        j,
    ] = train_median[j]


rdkit_kept_raw = (
    rdkit_imputed[
        :,
        keep_mask,
    ]
)


assert np.isfinite(
    rdkit_kept_raw
).all()


# ============================================================
# 12) TRAIN-ONLY SCALER
# ============================================================

scaler = StandardScaler()

scaler.fit(
    rdkit_kept_raw[
        train_indices
    ]
)


rdkit_scaled = scaler.transform(
    rdkit_kept_raw
).astype(
    np.float32
)


assert np.isfinite(
    rdkit_scaled
).all()


# Strict scaler QA
np.testing.assert_allclose(
    rdkit_scaled[
        train_indices
    ].mean(axis=0),
    0.0,
    atol=2e-5,
)

np.testing.assert_allclose(
    rdkit_scaled[
        train_indices
    ].std(axis=0),
    1.0,
    atol=2e-4,
)


print("\n" + "=" * 90)
print("TRAIN-ONLY RDKIT FIT COMPLETE")
print("=" * 90)

print(
    "Original canonical descriptors:",
    n_desc,
)

print(
    "Retained using Train only:",
    len(kept_names),
)

print(
    "Dropped using Train only:",
    len(dropped_names),
)

print(
    "Dropped names:",
    dropped_names,
)


# ============================================================
# 13) CREATE THREE CONTROLLED FEATURE MATRICES
# ============================================================

X_morgan = (
    morgan.astype(
        np.float32
    )
)

X_rdkit = (
    rdkit_scaled.astype(
        np.float32
    )
)

X_both = np.concatenate(
    [
        X_morgan,
        X_rdkit,
    ],
    axis=1,
).astype(
    np.float32
)


assert X_morgan.shape == (
    1705,
    2048,
)

assert X_rdkit.shape == (
    1705,
    len(kept_names),
)

assert X_both.shape == (
    1705,
    2048 + len(kept_names),
)


# ============================================================
# 14) SAVE FEATURE CACHES
# ============================================================

cache_paths = {
    "morgan_only":
        OUT / "molecular_features_morgan_only.npz",

    "rdkit_only":
        OUT / "molecular_features_rdkit_only.npz",

    "morgan_rdkit":
        OUT / "molecular_features_morgan_rdkit.npz",
}


np.savez_compressed(
    cache_paths["morgan_only"],
    x=X_morgan,
    all_drugs=all_drugs,
    train_drug_indices=train_indices,
)


np.savez_compressed(
    cache_paths["rdkit_only"],
    x=X_rdkit,
    all_drugs=all_drugs,
    train_drug_indices=train_indices,
)


np.savez_compressed(
    cache_paths["morgan_rdkit"],
    x=X_both,
    all_drugs=all_drugs,
    train_drug_indices=train_indices,
)


# ============================================================
# 15) DESCRIPTOR AUDIT TABLE
# ============================================================

descriptor_audit = pd.DataFrame({
    "descriptor":
        canonical_descriptor_names,

    "train_finite_count":
        train_finite_count,

    "train_missing_count":
        len(train_indices)
        -
        train_finite_count,

    "train_median":
        train_median,

    "train_variance_after_impute":
        train_variance,

    "retained":
        keep_mask.astype(int),

    "raw_all_drug_failure_count":
        descriptor_failure_counts,
})


descriptor_audit.to_csv(
    OUT / "rdkit_train_only_descriptor_audit.csv",
    index=False,
)


pd.DataFrame({
    "descriptor": kept_names,
    "scaler_mean_train_only": scaler.mean_,
    "scaler_scale_train_only": scaler.scale_,
}).to_csv(
    OUT / "rdkit_train_only_scaler.csv",
    index=False,
)


# ============================================================
# 16) SAVE UNSEEN DRUG AUDIT
# ============================================================

unseen_rows = []

for split_name, values in [
    ("calibration", cal_unseen),
    ("test", test_unseen),
]:

    for drug in sorted(values):

        unseen_rows.append({
            "split": split_name,
            "drug": drug,
            "global_index": drug_to_idx[drug],
        })


pd.DataFrame(
    unseen_rows
).to_csv(
    OUT / "heldout_drugs_not_in_train.csv",
    index=False,
)


# ============================================================
# 17) FINAL METADATA
# ============================================================

metadata = {

    "purpose":
        "Controlled feature ablation with Train-only "
        "RDKit preprocessing",

    "old_feature_cache_reused":
        False,

    "old_cache_role":
        "all_drugs ordering only",

    "historical_meta_role":
        "canonical descriptor names and Morgan definition only",

    "split_rows": {
        name: len(df)
        for name, df in splits.items()
    },

    "total_drugs":
        len(all_drugs),

    "train_fit_drugs":
        len(train_drugs),

    "calibration_unseen_drugs":
        len(cal_unseen),

    "test_unseen_drugs":
        len(test_unseen),

    "rdkit_version_current":
        rdBase.rdkitVersion,

    "rdkit_version_historical":
        historical_meta.get(
            "rdkit_version"
        ),

    "rdkit_canonical_count":
        n_desc,

    "rdkit_retained_train_only":
        len(kept_names),

    "rdkit_dropped_train_only":
        dropped_names,

    "rdkit_imputation":
        "Per-descriptor median fitted on Train drugs only",

    "rdkit_selection":
        "Train variance after Train-median imputation > 1e-12",

    "rdkit_scaler":
        "StandardScaler fitted on Train drugs only",

    "morgan_radius":
        MORGAN_RADIUS,

    "morgan_bits":
        MORGAN_BITS,

    "morgan_use_chirality":
        False,

    "morgan_use_features":
        False,

    "feature_shapes": {
        "morgan_only":
            list(X_morgan.shape),

        "rdkit_only":
            list(X_rdkit.shape),

        "morgan_rdkit":
            list(X_both.shape),
    },

    "source_sha256": {
        "train":
            sha256_file(TRAIN_PATH),

        "calibration":
            sha256_file(CAL_PATH),

        "test":
            sha256_file(TEST_PATH),

        "old_cache":
            sha256_file(OLD_CACHE),

        "old_meta":
            sha256_file(OLD_META),
    },

    "new_cache_sha256": {
        mode:
            sha256_file(path)
        for mode, path
        in cache_paths.items()
    },
}


save_json(
    metadata,
    OUT / "feature_ablation_metadata.json",
)


# ============================================================
# 18) README
# ============================================================

README = f"""
FEATURE ABLATION — TRAIN-ONLY PREPROCESSING
===========================================

This package prepares three controlled molecular feature sets:

1. Morgan-only
2. RDKit-only
3. Morgan + RDKit

Original Interaction splits are unchanged.

Train rows:
134,249

Calibration rows:
28,768

Test rows:
28,768

Unique drugs:
1,705

Train-fit drugs:
{len(train_drugs)}

Calibration drugs absent from Train:
{len(cal_unseen)}

Test drugs absent from Train:
{len(test_unseen)}

Morgan:
radius = 2
bits = 2048
chirality = False
feature invariants = False

RDKit:
Canonical historical descriptor list = {n_desc}
Train-only retained descriptors = {len(kept_names)}

RDKit selection, imputation, and scaling are fitted using
Train drugs only. Calibration and Test never affect these fitted
parameters.

The historical feature cache is NOT overwritten and its feature
values are NOT used for this new experiment.

The historical cache is used only to preserve exact drug indexing.
The historical metadata is used only to preserve descriptor names
and Morgan configuration.

NO TRAINING is performed by this preparation cell.
""".strip()


(
    OUT / "README.txt"
).write_text(
    README,
    encoding="utf-8",
)


# ============================================================
# 19) ZIP
# ============================================================

ZIP_PATH = Path(
    shutil.make_archive(
        str(
            OUT.parent /
            "FEATURE_ABLATION_TRAIN_ONLY_PREPROCESSING"
        ),
        "zip",
        root_dir=OUT,
    )
)


with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as z:

    assert z.testzip() is None


# ============================================================
# 20) FINAL REPORT
# ============================================================

print("\n" + "=" * 90)
print(" FEATURE PREPARATION COMPLETE")
print("=" * 90)

print(
    "Train fit drugs:",
    len(train_drugs),
)

print(
    "All drugs:",
    len(all_drugs),
)

print(
    "CAL unseen drugs:",
    len(cal_unseen),
)

print(
    "TEST unseen drugs:",
    len(test_unseen),
)

print(
    "RDKit canonical descriptors:",
    n_desc,
)

print(
    "RDKit retained Train-only:",
    len(kept_names),
)

print(
    "Morgan-only:",
    X_morgan.shape,
)

print(
    "RDKit-only:",
    X_rdkit.shape,
)

print(
    "Morgan+RDKit:",
    X_both.shape,
)

print(
    "NaN Morgan:",
    int(
        np.isnan(X_morgan).sum()
    ),
)

print(
    "NaN RDKit:",
    int(
        np.isnan(X_rdkit).sum()
    ),
)

print(
    "NaN Both:",
    int(
        np.isnan(X_both).sum()
    ),
)

print("\nSaved to:")
print(OUT)

print("\nZIP:")
print(ZIP_PATH)

print("\n OLD CACHE UNTOUCHED")
print(" TRAIN-ONLY DESCRIPTOR SELECTION")
print(" TRAIN-ONLY IMPUTATION")
print(" TRAIN-ONLY SCALING")
print(" NO TRAINING PERFORMED")